In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv("../data/processed/podcast_trilingual_embeddings.csv")
df.head()
df.columns

Index(['start', 'end', 'en', 'he', 'ar', 'en_embedding', 'he_embedding',
       'ar_embedding'],
      dtype='object')

In [4]:
import ast
import numpy as np

def parse_embedding(x):
    if isinstance(x, str):
        # remove brackets
        x = x.replace("[", "").replace("]", "")
        # split by whitespace
        return np.array(x.split(), dtype=float)
    return np.array(x, dtype=float)

E = np.vstack(df["en_embedding"].apply(parse_embedding).to_numpy())
H = np.vstack(df["he_embedding"].apply(parse_embedding).to_numpy())
A = np.vstack(df["ar_embedding"].apply(parse_embedding).to_numpy())

print("English:", E.shape)
print("Hebrew:", H.shape)
print("Arabic:", A.shape)

English: (1735, 150)
Hebrew: (1735, 150)
Arabic: (1735, 150)


In [6]:
def project_and_residual(X_foreign, X_english):
    # Finds W such that English @ W ≈ Foreign
    W, _, _, _ = np.linalg.lstsq(X_english, X_foreign, rcond=None)

    X_projected = X_english @ W
    X_residual = X_foreign - X_projected

    return X_projected, X_residual

H_projected, H_residual = project_and_residual(H, E)
A_projected, A_residual = project_and_residual(A, E)

print("Hebrew residual:", H_residual.shape)
print("Arabic residual:", A_residual.shape)

Hebrew residual: (1735, 150)
Arabic residual: (1735, 150)


In [7]:
np.save("../data/processed/hebrew_residuals.npy", H_residual)
np.save("../data/processed/arabic_residuals.npy", A_residual)

np.save("../data/processed/hebrew_projected.npy", H_projected)
np.save("../data/processed/arabic_projected.npy", A_projected)

print("Saved residual and projected embeddings.")

Saved residual and projected embeddings.


In [8]:
import os

print(os.path.exists("../data/processed/hebrew_residuals.npy"))
print(os.path.exists("../data/processed/arabic_residuals.npy"))

True
True
